## Fourth step

In `fourth_step.ipynb`, we will use the records filtered (or not) in `third_step.ipynb` and filter them based on barcodes. Each herbarium specimen must be assigned a barcode for unique identification. If we use specimens without barcodes, we will be unable to identify them later if necessary. Therefore, we will remove all records that lack a barcode.

Following the same convention as the previous step, the new field will be named `barcode_upd`, since the original field is `barcode`.

In [ ]:
from library import *

specieslink, db_config = configure()

In [ ]:
field_input = input("specify the name you want the new field to have (do not create the field manually when running via the pipeline!)").strip()
original_input = "barcode"
table = "biodiversity_records"

try:
    conn = mysql_conn.connect(**db_config)
    cursor = conn.cursor()

    sql = f"ALTER TABLE {table} ADD COLUMN {field_input} TEXT"

    cursor.execute(sql)
    conn.commit()

    print(f"field '{field_input}' created with success in table '{table}'")

    cursor.close()
    conn.close()  
except Exception as e:
    print(f"field '{field_input}' already exists or error while creating: {e}")

conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()

sql = f"""
UPDATE {table}
SET {field_input} = {original_input}
WHERE {original_input} IS NOT NULL
"""

cursor.execute(sql)
conn.commit()

cursor.close()
conn.close()

In [ ]:
conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()

sql = """
SELECT
    COUNT(*) AS total_records,
    SUM(country_upd IS NOT NULL) AS count_country,
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL) AS count_country_n_state,
    SUM(country_upd IS NOT NULL AND stateprovince_upd IS NOT NULL AND barcode_upd IS NOT NULL) as count_country_state_n_barcode
FROM biodiversity_records
"""

cursor.execute(sql)
total_records, records_country, records_country_state, records_country_state_barcode = cursor.fetchone()

cursor.close()
conn.close()

In [ ]:
plt.figure(figsize=(6, 4))

x = [0]

plt.bar(
    x,
    [total_records],
    alpha=0.5,
    width=0.15,
    label='total'
)

plt.bar(
    x,
    [records_country],
    alpha=0.5,
    width=0.15,
    label='country_upd'
)

plt.bar(
    x,
    [records_country_state],
    width=0.15,
    label='country_upd and stateprovince_upd'
)

plt.bar(
    x,
    [records_country_state_barcode],
    width=0.15,
    label='country_upd, stateprovince_upd and barcode_upd'
)

plt.xlim(-0.2, 0.3)
plt.ylabel('total records')
plt.title('sample of records to be used')
plt.legend()

plt.text(0, total_records, str(total_records), ha='center', va='bottom')
plt.text(0, records_country, str(records_country), ha='center', va='bottom')
plt.text(0, records_country_state, str(records_country_state), ha='center', va='bottom')
plt.text(0, records_country_state_barcode, str(records_country_state_barcode), ha='center', va='bottom')

plt.tight_layout()
plt.show()